In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time,math

In [ ]:
def scaled_dot_product_attention(q,k,v,mask=None):
  #q:[B,H,T,D_Q]
  #k:[B,H,t,D_K]
  #v:[B,H,T,D_V]
  d_k=q.size(-1)
  scores=q@k.transpose(-2,-1)/math.sqrt(d_k)
  if mask is not None:
    scores=scores.masked_fill(~mask,float('-inf'))
  att=F.softmax(scores,dim=-1)
  return scores@v,att

q=torch.randn(2,5,7,8)
k=torch.randn(2,5,7,8)
v=torch.randn(2,5,7,8)
scores,_=scaled_dot_product_attention(q,k,v)
print(scores)


tensor([[[[-6.9604e-01,  9.2763e-01,  2.0872e+00,  1.2400e-01,  2.0584e-02,
            3.1056e+00,  1.4834e+00,  1.7994e+00],
          [-1.1051e+00,  3.4261e-01, -2.3214e+00,  1.0864e-01,  2.6016e+00,
            8.8104e-01, -3.2156e+00, -1.1868e+00],
          [-3.7816e+00,  4.5887e+00,  1.7293e-01, -1.2425e+00,  2.5102e+00,
           -3.1045e+00, -2.2702e+00, -1.3179e+00],
          [-3.6217e+00,  6.9169e+00,  5.3119e+00, -6.7795e-01, -2.2783e-01,
           -2.4228e+00,  5.6706e-01, -3.0338e-01],
          [ 2.1084e-01, -7.3086e-01, -1.0254e+00,  1.1552e+00, -1.3187e+00,
            1.5521e+00,  9.4142e-01, -2.6073e-01],
          [ 2.8292e-01,  1.9736e+00,  2.9117e+00, -1.4557e-02,  1.8907e+00,
           -2.7990e+00, -4.5630e+00, -3.2462e+00],
          [-3.1512e+00,  2.1125e+00, -3.0181e+00, -1.0155e+00,  1.2211e-01,
           -2.0832e+00,  1.2918e+00,  7.0639e-01]],

         [[-4.4065e-01,  8.0064e-01,  6.4749e-01, -1.9906e-01,  1.9939e+00,
            1.4594e+00,  1.1207e+

In [ ]:
def my_softmax(x,dim=-1):
  max_val=torch.max(x,dim=dim,keepdim=True).values
  print("max_val:",max_val)
  exp_x=torch.exp(x-max_val)
  print(x-max_val)
  print("exp_x:",exp_x)
  sum_exp=torch.sum(exp_x,dim=dim,keepdim=True)
  print("sum_exp:",sum_exp)
  return exp_x/sum_exp
x=torch.tensor([[1,2],[3,4]])
out=my_softmax(x)
print(out)



max_val: tensor([[2],
        [4]])
tensor([[-1,  0],
        [-1,  0]])
exp_x: tensor([[0.3679, 1.0000],
        [0.3679, 1.0000]])
sum_exp: tensor([[1.3679],
        [1.3679]])
tensor([[0.2689, 0.7311],
        [0.2689, 0.7311]])


In [ ]:
def my_softmax_2(x,dim=-1):
  exp_x=torch.exp(x)
  sum_exp=torch.sum(exp_x,dim=dim,keepdim=True)
  return exp_x/sum_exp
x=torch.tensor([[1,2],[3,4]])
print(my_softmax_2(x))

tensor([[0.2689, 0.7311],
        [0.2689, 0.7311]])


In [ ]:
import torch

def softmax_naive(x, dim=-1):
    e = torch.exp(x)
    return e / e.sum(dim=dim, keepdim=True)

def softmax_stable(x, dim=-1):
    e = torch.exp(x - x.max(dim=dim, keepdim=True).values)
    return e / e.sum(dim=dim, keepdim=True)

# 1) 上溢：数值很大
x = torch.tensor([[1000., 1001.]])
print(torch.exp(x))          # tensor([[inf, inf]])
print(softmax_naive(x))      # tensor([[nan, nan]])  因为 inf/inf
print(softmax_stable(x))     # tensor([[0.2689, 0.7311]])  正确

# 2) 下溢：数值很负
x = torch.tensor([[-1000., -1001.]])
print(torch.exp(x))          # tensor([[0., 0.]])
print(softmax_naive(x))      # tensor([[nan, nan]])  因为 0/0
print(softmax_stable(x))     # tensor([[0.7311, 0.2689]])  正确

# 3) 临界点：float32 大约在 exp(88.7) 就溢出
print(torch.exp(torch.tensor(88.)))   # 1.6516e+38，还能表示
print(torch.exp(torch.tensor(89.)))   # inf

tensor([[inf, inf]])
tensor([[nan, nan]])
tensor([[0.2689, 0.7311]])
tensor([[0., 0.]])
tensor([[nan, nan]])
tensor([[0.7311, 0.2689]])
tensor(1.6516e+38)
tensor(inf)


In [ ]:
#完整的放缩点击包含softmax的完整代码：
def my_softmax_3(x,dim=-1):
  max_val=torch.max(x,dim=dim,keepdim=True).values
  exp_x=torch.exp(x-max_val)
  return exp_x/torch.sum(exp_x,dim=dim,keepdim=True)
def scaled_dot_product_attention_1(q,k,v,mask=None):
  #q=[B,H,t,d_k],k=[B,H,t,d_k],v=[B,H,t,d_v]
  d_k=q.size(-1)
  scores=q@k.transpose(-2,-1)/math.sqrt(d_k)
  if mask is not None:
    scores=scores.masked_fill(~mask,float("-inf"))
  att=my_softmax_3(scores)
  return att@v,att
q=torch.randn(2,5,7,8)
k=torch.randn(2,5,7,8)
v=torch.randn(2,5,7,9)
T=7
mask=torch.tril(torch.ones(T,T,dtype=bool))
print("mask:",mask)
out,att=scaled_dot_product_attention_1(q,k,v)
print("out:",out)
print("att:",att)
mask_out,mask_att=scaled_dot_product_attention_1(q,k,v,mask)
print("causal_out:",mask_out)
print("causal_att",mask_att)

mask: tensor([[ True, False, False, False, False, False, False],
        [ True,  True, False, False, False, False, False],
        [ True,  True,  True, False, False, False, False],
        [ True,  True,  True,  True, False, False, False],
        [ True,  True,  True,  True,  True, False, False],
        [ True,  True,  True,  True,  True,  True, False],
        [ True,  True,  True,  True,  True,  True,  True]])
out: tensor([[[[ 4.2718e-01, -6.9729e-02, -8.4608e-02, -2.4012e-01, -4.3338e-01,
            8.6238e-02,  8.8591e-01,  1.6386e-01, -4.4353e-01],
          [ 2.5962e-01,  3.1067e-01,  5.4927e-01, -7.7428e-01, -4.0926e-01,
            9.5378e-01,  1.2826e+00, -2.8991e-01, -2.6512e-01],
          [ 6.3327e-01,  6.7301e-02,  4.0744e-01,  2.4259e-02, -2.0021e-01,
           -6.0653e-01,  5.6577e-01, -2.6275e-01, -4.7621e-01],
          [ 5.0917e-01,  5.9981e-01, -1.2778e-01, -4.9638e-01, -2.1097e-01,
           -1.0213e-01,  6.2648e-01, -3.1397e-02, -6.4537e-01],
          [ 4.7

In [ ]:
class MutleHeadAttention(nn.Module):
  def __init__(self,d_model,n_head,drop_out=0.01) -> None:
    super().__init__()
    assert d_model%n_head==0
    self.d_model=d_model
    self.h=n_head
    self.dk=d_model//n_head
    self.W_q=nn.Linear(d_model,d_model)
    self.W_k=nn.Linear(d_model,d_model)
    self.W_v=nn.Linear(d_model,d_model)
    self.W_o=nn.Linear(d_model,d_model)
    self.dropout=nn.Dropout(drop_out)


  def split(self,x):
    B,T,_=x.shape
    return x.view(B,T,self.h,self.dk).transpose(1,2)

  def forward(self,x_q,x_kv,mask=None):
    q=self.split(self.W_q(x_q))
    k=self.split(self.W_k(x_kv))
    v=self.split(self.W_v(x_kv))
    out,self.lastatt=scaled_dot_product_attention_1(q,k,v,mask)
    out=self.dropout(out)
    B,H,T,dk=out.shape
    out=out.transpose(1,2).contiguous().view(B,T,H*dk)
    return self.W_o(out)


mha=MutleHeadAttention(32,4)
mha.eval() # Disable dropout for deterministic evaluation
x=torch.randn(2,6,32)
causal=torch.tril(torch.ones(6,6,dtype=bool))[None, None]
y1=mha(x,x,causal)
x2=x.clone();x2[:,4:]=torch.randn(2,2,32)
y2=mha(x2,x2,causal)
print(y1.shape)
assert torch.allclose(y1[:, :4], y2[:, :4], atol=1e-6)
print("causality OK")

torch.Size([2, 6, 32])
causality OK


In [ ]:
class multiheadattention(nn.Module):
  def __init__(self,d_model,n_head,drop_out=0.01) -> None:
    super().__init__()
    assert d_model%n_head==0
    self.d_model=d_model
    self.h=n_head
    self.dk=d_model//n_head
    self.W_q=nn.Linear(d_model,d_model)
    self.W_k=nn.Linear(d_model,d_model)
    self.W_v=nn.Linear(d_model,d_model)
    self.W_o=nn.Linear(d_model,d_model)
    self.dropout=nn.Dropout(drop_out)

  def split(self,x):
    B,T,_=x.shape
    x=x.view(B,T,self.h,self.dk).transpose(1,2)
    return x

  def forward(self,q,k,v,mask=None):
    q=self.split(self.W_q(q))
    k=self.split(self.W_k(k))
    v=self.split(self.W_v(v))
    out,self.lastatt=scaled_dot_product_attention_1(q,k,v,mask)
    out=self.dropout(out)
    B,H,T,dk=out.shape
    out=out.transpose(1,2).contiguous().view(B,T,H*dk)
    return self.W_o(out)

mha1=multiheadattention(32,4).eval()
x=torch.randn(2,6,32)
causal=torch.tril(torch.ones(6,6,dtype=bool))
y1=mha1(x,x,x,causal)
print(y1.shape)
x2=x.clone();x2[:,4:]=torch.randn(2,2,32)
y2=mha1(x2,x2,x2,causal)
assert torch.allclose(y1[:,:4],y2[:,:4],atol=1e-6)
print("causality OK")

torch.Size([2, 6, 32])
causality OK


In [ ]:
#位置编码
class SinusoidalPE(nn.Module):
  def __init__(self,d_model,max_len=2048) -> None:
    super().__init__()
    pe=torch.zeros(max_len,d_model)
    pos=torch.arange(max_len).unsqueeze(1).float()
    div=torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model))
    pe[:,0::2]=torch.sin(pos*div)
    pe[:,1::2]=torch.cos(pos*div)
    self.register_buffer("pe",pe)
  def forward(self,x):
    x=x+self.pe[:x.size(1)]
    return x

pe=SinusoidalPE(64)
print(pe.pe.shape)
print(pe.pe)
print(pe)


torch.Size([2048, 64])
tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  ...,  1.0000e+00,
          0.0000e+00,  1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  6.8156e-01,  ...,  1.0000e+00,
          1.3335e-04,  1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  9.9748e-01,  ...,  1.0000e+00,
          2.6670e-04,  1.0000e+00],
        ...,
        [ 1.7590e-01, -9.8441e-01,  4.2275e-01,  ...,  9.3460e-01,
          2.6934e-01,  9.6305e-01],
        [-7.3331e-01, -6.7989e-01,  9.2701e-01,  ...,  9.3454e-01,
          2.6947e-01,  9.6301e-01],
        [-9.6832e-01,  2.4972e-01,  9.3397e-01,  ...,  9.3448e-01,
          2.6959e-01,  9.6297e-01]])
SinusoidalPE()


In [ ]:
class sinusoidialPE1(nn.Module):
  def __init__(self,d_model,max_len=4096) -> None:
    super().__init__()
    pe=torch.zeros(max_len,d_model)
    pos=torch.arange(max_len).unsqueeze(1).float()
    div=torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model))
    pe[:,0::2]=torch.sin(pos*div)
    pe[:,1::2]=torch.cos(pos*div)
    self.register_buffer("pe",pe)

  def forward(self,x):
    return x+pe[:x.size(1)]
PE1=sinusoidialPE1(64)
print(PE1.pe.shape)




torch.Size([4096, 64])


In [ ]:
#FFN
class FeedForward(nn.Module):
  def __init__(self,d_model,d_dff,dropout=0.01) -> None:
    super().__init__()
    self.net=nn.Sequential(nn.Linear(d_model,d_dff),nn.GELU(),nn.Linear(d_dff,d_model),nn.Dropout(dropout))
  def forward(self,x):return self.net(x)

In [ ]:
#EncoderLayer
class EncoderLayer(nn.Module):
  def __init__(self,d_model,n_head,d_dff,drop_out=0.01) -> None:
    super().__init__()
    self.att=MutleHeadAttention(d_model,n_head,drop_out)
    self.ffn=FeedForward(d_model,d_dff,drop_out)
    self.lay1=nn.LayerNorm(d_model)
    self.lay2=nn.LayerNorm(d_model)
    self.dropout=nn.Dropout(drop_out)


  def forward(self,x,src_mask):
    x=self.lay1(x)
    x=x+self.dropout(self.att(x,x,src_mask))
    x=self.lay2(x)
    x=x+self.dropout(self.ffn(x))
    return x

In [ ]:
#DecoderLayer
class DecoderLayer(nn.Module):
  def __init__(self,d_model,n_head,d_dff,drop_out=0.01) -> None:
    super().__init__()
    self.self_attn=MutleHeadAttention(d_model,n_head,drop_out)
    self.cross_attn=MutleHeadAttention(d_model,n_head,drop_out)
    self.ffn=FeedForward(d_model,d_dff,drop_out)
    self.ly1=nn.LayerNorm(d_model)
    self.ly2=nn.LayerNorm(d_model)
    self.ly3=nn.LayerNorm(d_model)
    self.dropout=nn.Dropout(drop_out)
  def forward(self,x,memory,tgt_mask,src_mask):
    x=x+self.dropout(self.self_attn(self.ly1(x),self.ly1(x),tgt_mask))
    x=self.ly2(x)
    x=x+self.dropout(self.cross_attn(x,memory,src_mask))
    x=x+self.dropout(self.ffn(self.ly3(x)))
    return x


In [ ]:
#Transformer
class Transformer(nn.Module):
  def __init__(self,src_voc,tgt_voc,d_model=128,n_head=4,drop_out=0.01,pad_id=0,max_len=2048,n_layer=3,d_dff=512) -> None:
    super().__init__()
    self.d_model,self.padId=d_model,pad_id
    self.PE=SinusoidalPE(d_model,max_len)
    self.ln1=nn.LayerNorm(d_model)
    self.ln2=nn.LayerNorm(d_model)
    self.src_emb=nn.Embedding(src_voc,d_model,padding_idx=pad_id)
    self.tgt_emb=nn.Embedding(tgt_voc,d_model,padding_idx=pad_id)
    self.dropout=nn.Dropout(drop_out)
    self.enc=nn.ModuleList(EncoderLayer(d_model,n_head,d_dff,drop_out) for _ in range(n_layer))
    self.dec=nn.ModuleList(DecoderLayer(d_model,n_head,d_dff,drop_out) for _ in range(n_layer))
    self.out=nn.Linear(d_model,tgt_voc)

  def make_mask(self,src_voc,tgt_voc):
    src_mask=(src_voc!=self.padId)[:,None,None,:]
    Tt=tgt_voc.size(1)
    causal=torch.tril(torch.ones(Tt,Tt,dtype=bool,device=tgt_voc.device))
    tgt_mask=(tgt_voc!=self.padId)[:,None,None,:]&causal
    return src_mask,tgt_mask

  def encode(self,src,src_mask):
    x=self.dropout(self.PE(self.src_emb(src)*math.sqrt(self.d_model)))
    for layer in self.enc:x=layer(x,src_mask)
    return self.ln1(x)

  def decode(self,tgt,memory,src_mask,tgt_mask):
    x=self.dropout(self.PE(self.tgt_emb(tgt)*math.sqrt(self.d_model)))
    for layer in self.dec:x=layer(x,memory,tgt_mask,src_mask)
    return self.out(self.ln2(x))

  def forward(self,src,tgt):
    src_mask,tgt_mask=self.make_mask(src,tgt)
    out=self.decode(tgt,self.encode(src,src_mask),src_mask,tgt_mask)
    return out

model=Transformer(20,20)
src=torch.randint(3,20,(2,8))
tgt=torch.randint(3,20,(2,9))
print(model(src, tgt).shape, "params:", sum(p.numel() for p in model.parameters()))

torch.Size([2, 9, 20]) params: 1396756
